SCENARIO: “Hospital Smart Assistant System”
🏥 Background Story
A large hospital deploys an AI-powered patient assistant.
👉 Patients can ask:
- “What is my appointment schedule?”
- “What are my latest test results?”
👉 Instead of calling reception or logging into multiple portals,
👉 AI fetches it instantly, providing secure, real-time updates.

In [ ]:
# ======================================
# STEP 1: Install Libraries
# ======================================
!pip install groq gradio nest_asyncio


# ======================================
# STEP 2: Load API Key using os.environ
# ======================================
import os

# 🔑 Replace with your NEW Groq API key
os.environ["GROQ_API_KEY"] = "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"

from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))


# ======================================
# STEP 3: MOCK TOOLS (Hospital MCP Tools)
# ======================================
import asyncio
import nest_asyncio

# Keep this because you said don't touch the structure
nest_asyncio.apply()

# --------------------------------------
# Compatibility patch for Gradio/Uvicorn
# This fixes: unexpected keyword argument 'loop_factory'
# --------------------------------------
_original_asyncio_run = asyncio.run

def compatible_asyncio_run(main, *, debug=None, loop_factory=None):
    return _original_asyncio_run(main)

asyncio.run = compatible_asyncio_run


patients = {
    "P101": {
        "name": "Rahul Sharma",
        "age": 29,
        "gender": "Male",
        "appointments": [
            {"date": "2026-03-30", "time": "10:00 AM", "department": "Cardiology", "doctor": "Dr. Mehta"},
            {"date": "2026-04-02", "time": "03:30 PM", "department": "Radiology", "doctor": "Dr. Arora"}
        ],
        "test_results": [
            {"test": "Blood Sugar", "result": "96 mg/dL", "status": "Normal"},
            {"test": "Cholesterol", "result": "210 mg/dL", "status": "Slightly High"}
        ],
        "profile": {
            "blood_group": "B+",
            "allergies": "Penicillin",
            "chronic_condition": "Mild Hypertension"
        }
    },
    "P102": {
        "name": "Priya Verma",
        "age": 35,
        "gender": "Female",
        "appointments": [
            {"date": "2026-03-31", "time": "11:30 AM", "department": "Neurology", "doctor": "Dr. Khanna"}
        ],
        "test_results": [
            {"test": "Hemoglobin", "result": "11.8 g/dL", "status": "Slightly Low"},
            {"test": "Vitamin D", "result": "18 ng/mL", "status": "Low"}
        ],
        "profile": {
            "blood_group": "O+",
            "allergies": "None",
            "chronic_condition": "None"
        }
    }
}


async def get_appointment_schedule(patient_id):
    await asyncio.sleep(1)
    if patient_id in patients:
        appointments = patients[patient_id]["appointments"]
        if not appointments:
            return "📅 No upcoming appointments found."

        text = "📅 Appointment Schedule:\n"
        for appt in appointments:
            text += (
                f"- Date: {appt['date']}, Time: {appt['time']}, "
                f"Department: {appt['department']}, Doctor: {appt['doctor']}\n"
            )
        return text.strip()
    return "❌ Patient not found."


async def get_latest_test_results(patient_id):
    await asyncio.sleep(1)
    if patient_id in patients:
        tests = patients[patient_id]["test_results"]
        if not tests:
            return "🧪 No recent test results found."

        text = "🧪 Latest Test Results:\n"
        for test in tests:
            text += (
                f"- Test: {test['test']}, Result: {test['result']}, Status: {test['status']}\n"
            )
        return text.strip()
    return "❌ Patient not found."


async def fetch_patient_profile(patient_id):
    await asyncio.sleep(1)
    if patient_id in patients:
        patient = patients[patient_id]
        profile = patient["profile"]
        return (
            f"👤 Patient Profile:\n"
            f"- Name: {patient['name']}\n"
            f"- Age: {patient['age']}\n"
            f"- Gender: {patient['gender']}\n"
            f"- Blood Group: {profile['blood_group']}\n"
            f"- Allergies: {profile['allergies']}\n"
            f"- Chronic Condition: {profile['chronic_condition']}"
        )
    return "❌ Patient not found."


# ======================================
# STEP 4: PARALLEL TOOL INVOCATION
# ======================================
async def parallel_patient_research(patient_id):
    results = await asyncio.gather(
        get_appointment_schedule(patient_id),
        get_latest_test_results(patient_id),
        fetch_patient_profile(patient_id),
        return_exceptions=True
    )

    appointments, tests, profile = results

    return {
        "appointments": appointments if not isinstance(appointments, Exception) else "Appointment data unavailable",
        "tests": tests if not isinstance(tests, Exception) else "Test data unavailable",
        "profile": profile if not isinstance(profile, Exception) else "Profile data unavailable"
    }


# ======================================
# STEP 5: CHAINED TOOL INVOCATION USING GROQ
# ======================================
def decide_intent(user_query):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
You are a hospital smart assistant intent classifier.

Classify the user's query into exactly one of these:
- appointment
- test_results
- profile
- full_summary

Rules:
- If the user asks about appointment, doctor visit, schedule, timing -> appointment
- If the user asks about reports, lab tests, results -> test_results
- If the user asks about patient details, profile, allergies, blood group -> profile
- If the user asks generally about health record, full details, complete summary -> full_summary

Only return one label.

User Query: {user_query}
"""
        }]
    )
    return response.choices[0].message.content.strip().lower()


def analyse_patient_data(text):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
Analyze this hospital data and give:
1. Key medical summary
2. Important observations
3. Appointment-related note
4. Risk flags if any
5. Simple patient-friendly explanation

Data:
{text}
"""
        }]
    )
    return response.choices[0].message.content


def generate_patient_report(analysis, patient_id):
    patient_name = patients.get(patient_id, {}).get("name", "Patient")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
Create a professional but simple hospital report for {patient_name}.

Use this analysis:
{analysis}

Keep it:
- clear
- structured
- patient-friendly
- concise but informative
"""
        }]
    )
    return response.choices[0].message.content


# ======================================
# STEP 6: FULL MCP PIPELINE
# ======================================
async def full_pipeline(patient_id, user_query):
    if patient_id not in patients:
        return "❌ Invalid Patient ID. Please enter P101 or P102."

    intent = decide_intent(user_query)
    data = await parallel_patient_research(patient_id)

    if intent == "appointment":
        combined_text = data["appointments"]

    elif intent == "test_results":
        combined_text = data["tests"]

    elif intent == "profile":
        combined_text = data["profile"]

    else:
        combined_text = f"""
{data['profile']}

{data['appointments']}

{data['tests']}
""".strip()

    analysis = analyse_patient_data(combined_text)
    report = generate_patient_report(analysis, patient_id)

    final_output = f"""
==============================
HOSPITAL SMART ASSISTANT OUTPUT
==============================

Patient ID: {patient_id}
Detected Intent: {intent}

RAW DATA:
{combined_text}

--------------------------------
AI ANALYSIS + PATIENT REPORT:
--------------------------------
{report}
"""
    return final_output.strip()


# ======================================
# STEP 7: NORMAL INPUT MODE
# ======================================
def run_normal_mode():
    print("🏥 Hospital Smart Assistant System")
    patient_id = input("Enter Patient ID (P101 / P102): ").strip()
    user_question = input("Ask your question: ").strip()

    result = asyncio.run(full_pipeline(patient_id, user_question))
    print("\n📋 FINAL OUTPUT:\n")
    print(result)


# ======================================
# STEP 8: GRADIO UI
# ======================================
import gradio as gr

def hospital_assistant_ui(patient_id, user_query):
    patient_id = patient_id.strip()
    user_query = user_query.strip()

    if not patient_id or not user_query:
        return "⚠️ Please enter both Patient ID and question."

    return asyncio.run(full_pipeline(patient_id, user_query))


with gr.Blocks() as demo:
    gr.Markdown("# 🏥 Hospital Smart Assistant System")
    gr.Markdown("""
Ask things like:
- What is my appointment schedule?
- What are my latest test results?
- Show my patient profile
- Give my complete health summary
""")

    patient_id_input = gr.Textbox(
        label="Enter Patient ID",
        placeholder="Example: P101 or P102"
    )

    query_input = gr.Textbox(
        label="Ask your question",
        placeholder="Example: What are my latest test results?"
    )

    output_box = gr.Textbox(
        label="Assistant Response",
        lines=24
    )

    submit_btn = gr.Button("Get Patient Details")

    submit_btn.click(
        fn=hospital_assistant_ui,
        inputs=[patient_id_input, query_input],
        outputs=output_box
    )


# ======================================
# STEP 9: RUN BOTH
# ======================================

# 1) Normal manual input mode
run_normal_mode()

# 2) Gradio real-time mode
demo.launch(share=True, debug=True)

🏥 Hospital Smart Assistant System

📋 FINAL OUTPUT:

HOSPITAL SMART ASSISTANT OUTPUT

Patient ID: P101
Detected Intent: appointment

RAW DATA:
📅 Appointment Schedule:
- Date: 2026-03-30, Time: 10:00 AM, Department: Cardiology, Doctor: Dr. Mehta
- Date: 2026-04-02, Time: 03:30 PM, Department: Radiology, Doctor: Dr. Arora

--------------------------------
AI ANALYSIS + PATIENT REPORT:
--------------------------------
**Hospital Report for Rahul Sharma**

**Patient Information:**
Name: Rahul Sharma

**Upcoming Appointments:**
1. **Cardiology Department**
   - Date: March 30, 2026
   - Time: 10:00 AM
   - Doctor: Dr. Mehta
2. **Radiology Department**
   - Date: April 2, 2026
   - Time: 3:30 PM
   - Doctor: Dr. Arora

**Important Notes:**
- Please arrive on time for both appointments.
- Follow any specific instructions provided by Dr. Mehta and Dr. Arora, such as preparation for radiology tests.
- Be prepared to discuss your medical history and current symptoms with both doctors.

**Addition